In [1]:
"""First, Train LAB Model (Required for Robustness Testing)"""

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split
import numpy as np
import os

print("="*70)
print("TRAINING LAB MODEL FIRST")
print("="*70)

# ============================================
# CONFIGURATION
# ============================================

class Config:
    EPOCHS = 5
    BATCH_SIZE = 128
    LR = 0.001
    WEIGHT_DECAY = 1e-4
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    DATA_DIR = "./data"

os.makedirs("outputs/checkpoints", exist_ok=True)
print(f"Device: {Config.DEVICE}")

# ============================================
# LAB TRANSFORM
# ============================================

class ToLAB:
    def __call__(self, img):
        img = np.array(img, dtype=np.float32) / 255.0
        r, g, b = img[..., 0], img[..., 1], img[..., 2]
        
        # RGB to XYZ
        r = np.where(r > 0.04045, ((r + 0.055) / 1.055) ** 2.4, r / 12.92)
        g = np.where(g > 0.04045, ((g + 0.055) / 1.055) ** 2.4, g / 12.92)
        b = np.where(b > 0.04045, ((b + 0.055) / 1.055) ** 2.4, b / 12.92)
        
        x = r * 0.4124564 + g * 0.3575761 + b * 0.1804375
        y = r * 0.2126729 + g * 0.7151522 + b * 0.0721750
        z = r * 0.0193339 + g * 0.1191920 + b * 0.9503041
        
        # XYZ to LAB
        x, y, z = x / 0.950456, y / 1.0, z / 1.088754
        x = np.where(x > 0.008856, x ** (1/3), (7.787 * x) + 16/116)
        y = np.where(y > 0.008856, y ** (1/3), (7.787 * y) + 16/116)
        z = np.where(z > 0.008856, z ** (1/3), (7.787 * z) + 16/116)
        
        l = (116 * y) - 16
        a = 500 * (x - y)
        b = 200 * (y - z)
        
        lab = np.stack([l/100.0, (a+128)/256.0, (b+128)/256.0], axis=2)
        return transforms.ToTensor()(lab)

# ============================================
# MODEL DEFINITION
# ============================================

import torchvision.models as models

class CNNBaseline(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.model = models.resnet18(weights=None)
        self.model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.model.maxpool = nn.Identity()
        self.model.fc = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        return self.model(x)

# ============================================
# DATA LOADING
# ============================================

# Training transform with augmentation
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    ToLAB(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Validation/Test transform (no augmentation)
transform_val = transforms.Compose([
    ToLAB(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

print("\nLoading CIFAR-10 dataset...")
train_dataset = torchvision.datasets.CIFAR10(
    root=Config.DATA_DIR, train=True, download=True, transform=transform_train
)
test_dataset = torchvision.datasets.CIFAR10(
    root=Config.DATA_DIR, train=False, download=True, transform=transform_val
)

# Split train/val
val_size = int(0.1 * len(train_dataset))
train_size = len(train_dataset) - val_size
train_subset, val_subset = random_split(train_dataset, [train_size, val_size])

train_loader = DataLoader(train_subset, batch_size=Config.BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=Config.BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=Config.BATCH_SIZE, shuffle=False)

print(f"Train samples: {train_size}")
print(f"Val samples: {val_size}")
print(f"Test samples: {len(test_dataset)}")

# ============================================
# TRAINING
# ============================================

model = CNNBaseline(num_classes=10).to(Config.DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=Config.LR, weight_decay=Config.WEIGHT_DECAY)

print("\n" + "="*60)
print("Training LAB Model...")
print("="*60)

best_val_acc = 0

for epoch in range(Config.EPOCHS):
    # Training
    model.train()
    train_correct = 0
    train_total = 0
    train_loss = 0
    
    for batch_idx, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(Config.DEVICE), labels.to(Config.DEVICE)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        _, predicted = outputs.max(1)
        train_total += labels.size(0)
        train_correct += predicted.eq(labels).sum().item()
        
        if (batch_idx + 1) % 100 == 0:
            print(f"  Batch {batch_idx+1}/{len(train_loader)} - Loss: {loss.item():.4f}")
    
    train_acc = 100. * train_correct / train_total
    avg_train_loss = train_loss / len(train_loader)
    
    # Validation
    model.eval()
    val_correct = 0
    val_total = 0
    val_loss = 0
    
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(Config.DEVICE), labels.to(Config.DEVICE)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
    
    val_acc = 100. * val_correct / val_total
    avg_val_loss = val_loss / len(val_loader)
    
    print(f"\nEpoch {epoch+1}/{Config.EPOCHS}:")
    print(f"  Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"  Val Loss:   {avg_val_loss:.4f} | Val Acc:   {val_acc:.2f}%")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "outputs/checkpoints/best_lab.pt")
        print(f"  ✅ Model saved (best val: {best_val_acc:.2f}%)")
    
    print("="*60)

# Test the model
model.load_state_dict(torch.load("outputs/checkpoints/best_lab.pt"))
model.eval()
test_correct = 0
test_total = 0

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(Config.DEVICE), labels.to(Config.DEVICE)
        outputs = model(inputs)
        _, predicted = outputs.max(1)
        test_total += labels.size(0)
        test_correct += predicted.eq(labels).sum().item()

test_acc = 100. * test_correct / test_total

print("\n" + "="*60)
print("✅ LAB MODEL TRAINING COMPLETE!")
print("="*60)
print(f"Best Validation Accuracy: {best_val_acc:.2f}%")
print(f"Final Test Accuracy: {test_acc:.2f}%")
print(f"Model saved to: outputs/checkpoints/best_lab.pt")

TRAINING LAB MODEL FIRST
Device: cpu

Loading CIFAR-10 dataset...
Train samples: 45000
Val samples: 5000
Test samples: 10000

Training LAB Model...
  Batch 100/352 - Loss: 1.4907
  Batch 200/352 - Loss: 1.5437
  Batch 300/352 - Loss: 1.2016

Epoch 1/5:
  Train Loss: 1.4570 | Train Acc: 46.83%
  Val Loss:   1.3189 | Val Acc:   54.80%
  ✅ Model saved (best val: 54.80%)
  Batch 100/352 - Loss: 0.9366
  Batch 200/352 - Loss: 0.8961
  Batch 300/352 - Loss: 0.9433

Epoch 2/5:
  Train Loss: 0.9739 | Train Acc: 65.62%
  Val Loss:   0.9758 | Val Acc:   64.80%
  ✅ Model saved (best val: 64.80%)
  Batch 100/352 - Loss: 0.7326
  Batch 200/352 - Loss: 0.9613
  Batch 300/352 - Loss: 0.8847

Epoch 3/5:
  Train Loss: 0.7805 | Train Acc: 72.86%
  Val Loss:   0.9682 | Val Acc:   66.64%
  ✅ Model saved (best val: 66.64%)
  Batch 100/352 - Loss: 0.7795
  Batch 200/352 - Loss: 0.7952
  Batch 300/352 - Loss: 0.6534

Epoch 4/5:
  Train Loss: 0.6661 | Train Acc: 77.12%
  Val Loss:   0.7148 | Val Acc:   76.00%